# Lexos Bootstrap Consensus Tree Tutorial

A Bootstrap Consensus Tree is a form of dendrogram produced by [Hierchical Agglomerative Clustering](https://scottkleinman.github.io/lexos/user_guide/cluster/hierarchical-agglomerative-clustering/). However, instead of building a single tree, it builds multiple trees by randomly sampling portions of your document-term matrix. It then finds the "consensus": the most consistently appearing relationships across all those individual trees. This can help increase your confidence in the "robustness" of your clustering results.

We'll begin by importing some data and creating a document-term matrix from a corpus of Shakespeare plays.

In [ ]:
# Python imports
from glob import glob
from lexos import DTM
from lexos.io.loader import Loader
from lexos import Tokenizer

# Load some text files and set their names
files = glob("bct_data/*.txt")  # Load all text files in the directory
loader = Loader()
loader.load(files)

# Tokenize the loaded documents
tokenizer = Tokenizer()
docs = list(tokenizer.make_docs(texts=loader.texts))
labels = loader.names

print(f"Loaded {len(docs)} documents with labels: {labels}")

# Create a Document-Term Matrix (DTM)
dtm = DTM()
dtm(docs=docs, labels=labels)

print(f"DTM created with {dtm.to_df().shape[1]} documents and {dtm.to_df().shape[0]} unique terms.")

### Generating the Bootstrap Consensus Tree (BCT):

When you create the BCT, you need to tell it how to build and combine the individual trees. Here are the key parameters you can use to adjust your analysis:

- `dtm`: The "linguistic spreadsheet" (`dtm`) that you created in the previous step.
- `metric`: This tells the tree how to measure the "distance" or dissimilarity between your documents. Shorter distances mean more similar documents. The default value "euclidean" is good for general comparisons but can be sensitive to the overall length of documents. For further discussion, see the documentation on [Hierchical Agglomerative Clustering](https://scottkleinman.github.io/lexos/user_guide/cluster/hierarchical-agglomerative-clustering/).
- `method`: The criterion used to join clusters to form larger branches and clusters in the tree. Possible values are "average" (the default), "single", "complete", and "ward". For further discussion, see the documentation on [Hierchical Agglomerative Clustering](https://scottkleinman.github.io/lexos/user_guide/cluster/hierarchical-agglomerative-clustering/).
- `cutoff`: This is a confidence threshold. Since a BCT is built from many individual "bootstrap" trees, a `cutoff` of `0.5` (50%) means that a specific grouping of documents (a branch on the tree) must appear in at least 50% of all the trees generated in each iteration to be considered reliable enough to show up in the final consensus tree. Higher `cutoff` values will result in a "sparser" tree, showing only the most robust and consistent relationships. Lower `cutoff` values will show more relationships, but some of these might be less statistically reliable.
- `iterations`: The number of "bootstrap resampling" rounds. In each round, Lexos takes a random 80% sample of the terms (columns) from your DTM and builds a tree from that sample. More iterations will make the consensus tree more statistically reliable and representative of the underlying relationships in your documents, as it averages out more variations. However, it will take longer to compute. Since fewer iterations will be faster, a lower number is good for quick testing or initial explorations. Setting `iterations` to 100 or higher is recommended for final research results if computation time allows.
- `replace`: This determines how the terms are sampled during each iteration. If set to "with", a term column can be selected multiple times within a single 80% sample (which allows for more randomness). The default setting "without" means that each term column can only be selected once per 80% sample (which is more stable).

Keep in mind that there is an element of randomness in the sampling of your data, which means that the results of successive runs will not be the same. Setting `iterations` to a higher value will cause multiple runs to converge on the same results. If you need to reproduce results exactly, set the `random_seed` value to a number of your choice. This will ensure that the bootstrapping process produces the same tree every time you run it.

In the example below, we will create a Bootstrap Consensus Tree with the default values, except that we will keep the number of iterations low for demonstration purposes. Feel free to change the commented out settings to see how they affect the results.

In [ ]:
# Import the BCT class
from lexos.cluster import BCT

bct = BCT(
    dtm=dtm,
    labels=labels,
    random_seed=42,                # For reproducibility
    iterations=10,                 # Set higher for more slower but more reliable results
#     metric="euclidean",          # Try "cosine" for stylistic comparisons, or "cityblock"
#     method="average",            # Try "ward" for compact clusters, or "complete"
#     cutoff=0.5,                  # Only show relationships present in at least 50% of trees
#     replace="without",           # Do not repeat terms in each sample
)

# Show the figure
bct.show()

The tree you see is a **dendrogram**, a type of tree diagram that shows hierarchical clustering. Each line is a **branch** on the the tree and connects either to another branch or to a **leaf** (a terminal branch). Each leaf represents an individual document and is labelled with that document's name. Leaves connected to other leaves form clusters, or **clades**, and these may be joined to other clades, forming "super-clusters" at the next level in the tree's hierarchy. In the rectangular layout, branch length indicates the **distance** or **dissimilarity** between the documents in a clade and those that form part of the same "super-cluster". Shorter branches mean that the documents or clusters are more similar to each other based on their linguistic features. Longer horizontal branches mean that the clades are further apart.

The following procedure is useful in reading the tree.

1. Start from the leaves (your document names) and move up the hierarchy towards the root of the tree. Look for the first merges.
2. As you move further up the hierarchy, you'll see larger branches forming, grouping together sets of documents or smaller clusters. These broader groupings represent larger patterns of similarity.

If you produce a tree that looks blank or empty, check your `cutoff`. If `cutoff` is too high (e.g., 0.9 or 1.0), it might be too strict, and no common groupings meet that threshold. Try lowering it (e.g., to 0.5 or 0.3). You an also check `iterations`. If it is very low, there might not be enough "sampling" to find a stable consensus. Finally, be sure to check that your `dtm` was created successfully and contains terms. If your documents are very short or very similar, the DTM might be sparse.

If your documents don't cluster the way you expect, experiment with the `metric` and `method` parameters, as well as the number of `iterations`.

### Customizing the Look of the Tree

There are a number of parameters for customizing the look of the diagram.

- `labels`: A list of labels to use for the leaves (endpoints) on your tree. The list should have the same order as the names in your DTM and will override any names associated with the DTM.
- `label_colors`: Sets the color for the document labels. If you set it to "auto", the colors will be chosen by the clade structure. You can also provide a dictionary where the keys are hexidecimal color codes and the value are lists of corresponding labels. If you do not set it, the default color is black.
- `title`: A title to place at the top of the plot.
- `figsize`: Optional figure size as (width, height) in inches. The default is `(10, 10)`.
- `label_fontsize`: Sets the font size for document labels.
- `font_family`: Sets the font family for the entire plot. The default is "sans-serif".

When you modify any visual settings, call `bct.show()` to redraw the figure using the existing consensus tree.

In the example below, we'll shrink the output, add a custom title, and set some custom label colors. We'll also include the default clustering and bootstrapping settings in case you want to try to adjust them. You will notice that we are not creating a new `BCT` object. We want to re-use our existing clustering.

In [ ]:
label_colors = {
    "#FF0000": ['titus-andronicus', 'antony-and-cleopatra'],
    "#00FF00": ['measure-for-measure', 'henry-iv-part-1'],
    "#0000FF": ['two-gentlemen-of-verona'],
}

# Set custom parameters for the BCT visualization
bct.title = "Custom Bootstrap Consensus Tree"
bct.figsize = (6, 6)  # Set the figure size (width, height) in inches
bct.label_fontsize = 10
bct.label_colors = label_colors

# Show the figure
bct.show()

### Fan Layout

A popular way to visualize bootstrap consensus trees is with a fan layout. You can produce one by setting the `layout` parameter to "fan" when you create your `BCT` instance (the default is "rectangular").

Note that in fan layouts, branch lengths are not significant since they are scaled to allow the labels to extend to the image bounds.

In [ ]:
bct = BCT(
    dtm=dtm,
    labels=labels,
    layout="fan",
    figsize=(6, 6),
    label_fontsize=10,
    label_colors="auto",
    iterations=10,
    random_seed=42,
    title="Bootstrap Consensus Tree (Fan Layout)",
)
bct.show()

Fan layouts have a few extra parameters you can use to customize the look:

- `linewidth`: The width of the tree lines (the default is 1.2).
- `min_leaf_len`: The minimum length of leaf branches (the default is 1.5).
- `internal_scale`: The scaling factor for internal brances (the default is 0.6).

In general, you only need to use these for slight modifications to the appearance of your tree. Otherwise, fan layouts work with all the same parameters as rectangular layouts.

### Changing `BCT` Settings

You can change any setting with commands like `bct.metric="ward"`, `bct.title="My New Title"`, or `bct.layout="fan"` and then calling `bct.show()`.

`BCT` attributes can be categorized either as "tree settings" (settings that affect the underlying tree structure) or "visual settings" (settings that affect the appearance of the figure output). If you wish to change the visual settings, you generally want to redraw the figure with the existing tree since this is much faster than rebuilding the entire consensus tree.

Depending on which settings you change, the following behaviors occur: 

- `bct.show()` always applies your current settings.
- Tree settings rebuild the consensus tree on the next `show()`: `dtm`, `metric`, `method`, `cutoff`, `iterations`, `replace`, `random_seed`.
- Visual settings redraw the figure using the existing tree: `layout`, `title`, `figsize`, `label_fontsize`, `fontfamily`, `label_colors`, `linewidth`, `min_leaf_len`, `internal_scale`, and `labels`.
- Use `show(layout="fan")` or `show(layout="rectangular")` only when you want to switch layouts.

In [ ]:
# Visual setting changes redraw the existing tree
bct.title = "Bootstrap Consensus Tree (Rectangular Layout)"
bct.label_colors = "auto"
bct.show(layout="rectangular")

# Tree-setting changes trigger a rebuild on the next show()
bct.random_seed = 7
bct.show()

### Saving Your Tree

Once you're happy with your Bootstrap Consensus Tree, you'll likely want to save it as an image file for reports, presentations, or simply for your records.

The `bct.save()` method allows you to do this easily. Just provide a file name, and it will save the image in the same directory where your Jupyter Notebook is located. You can choose different file formats by changing the extension (e.g., `.png`, `.jpg`, `.pdf`, `.svg`). PNG is generally a good choice for web or documents, while SVG provides a scalable vector graphic useful for high-quality printing.

In [ ]:
# Save the figure to a file
# bct.save("bct_test_result.png")